# EXP-06B — Factory e possíveis eventos de retirada (Privacy Pools v1)

**Objetivo:** complementar o EXP-06A sem repeti-lo: procurar *traces* de possível criação interna do pool e fazer uma triagem **amostral e limitada** de logs. **Não atribui rótulo de mixer**, não usa Random Forest, não treina, não altera EXP-05.

**Execução:** Ambiente de execução → Executar tudo; autorize Drive; resultado é salvo em uma **nova pasta** dentro de `results/`. Se uma fonte falhar, o relatório guardará `inconclusivo`, e não `ausência de implantação/saque`.

⚠️ A API Blockscout pode devolver apenas estado atual ou paginação parcial. Um evento cujo nome sugere retirada não prova por si só desvinculação entre depósito e saque.

In [ ]:
%pip -q install requests google-api-python-client google-auth-httplib2 google-auth-oauthlib
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from pathlib import Path
from datetime import datetime, timezone
import requests, subprocess, sys, hashlib, io, json
api=build('drive','v3')
base=Path('/content/EXP-06B'); base.mkdir(parents=True,exist_ok=True)
print('Drive autenticado; projeto EXP-06B iniciado.')

## 1. Recuperar entrada EXP-06A congelada e verificar integridade
O arquivo original deve ter o mesmo SHA-256 já auditado. O notebook não consulta as duas transações do EXP-06A novamente.

In [ ]:
input_id='1vAiHf6_ui9-SIrfuoQaIV2xWEFkALJYj'
original=base/'verificacao_exp06a.json'
req=api.files().get_media(fileId=input_id)
with original.open('wb') as target:
    dl=MediaIoBaseDownload(target,req);done=False
    while not done: _,done=dl.next_chunk()
sha_file=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
expected='f3e142a11e8a6d954d858c22df0dfa35b6a06e71234a75fa4cff03e9dadc3dda'
assert sha_file(original)==expected,'ERRO: EXP-06A original divergente — interromper'
print('EXP-06A original íntegro:',sha_file(original))

## 2. Obter script versionado no GitHub e conferir SHA-256
Não execute código de versões diferentes do experimento.

In [ ]:
commit='2b2a1f0e7c0f030f942a74acf7496ef8ff617108'
script_url=f'https://raw.githubusercontent.com/BorgesGusta/ethereum-mixer-detection/{commit}/scripts/exp06b_validar_criacao_saques.py'
r=requests.get(script_url,timeout=40);r.raise_for_status()
script=base/'exp06b_validar_criacao_saques.py';script.write_bytes(r.content)
expected_script_sha='17d2aab3ad5e55ddb63af78d3a3e5afba542661b57e8de35cc82f374c6865bf9'
assert sha_file(script)==expected_script_sha,'ERRO: script não corresponde ao commit congelado'
print('Script verificado:',sha_file(script))

## 3. Fazer apenas as consultas complementares
Consulta no máximo quatro páginas de *traces* da transação candidata de criação e duas páginas de logs do pool. **A ausência de um evento nessa amostra não é prova de inexistência de saques**.

In [ ]:
out=base/'saida'
subprocess.run([sys.executable,str(script),'--exp06a',str(original),'--saida',str(out)],check=True)
data=json.loads((out/'evidencias_exp06b.json').read_text(encoding='utf-8'))
print('Estado dos traces:',data['traces_candidata']['status'])
print('Criações internas do pool observadas:',len(data['traces_candidata']['traces_criacao_pool_sucesso']))
print('Estado da amostra de logs:',data['amostra_logs_pool']['status'])
print('Amostras de logs:',data['amostra_logs_pool']['total_logs_observados'])
print('Qualquer classificação funcional continua PENDENTE.')

## 4. Salvar em nova pasta do Drive oficial
O notebook não modifica o EXP-06A nem outras saídas. Arquivos gerados: `evidencias_exp06b.json` e `manifesto_exp06b.json`.

In [ ]:
parent='1iZKAeZPEb6JOlH8KfqsR6-wqI1ygmojI'
name='EXP-06B_traces_e_logs_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
folder=api.files().create(body={'name':name,'mimeType':'application/vnd.google-apps.folder','parents':[parent]},fields='id,name').execute()
for file in sorted(out.iterdir()):
    mime='application/json'
    saved=api.files().create(body={'name':file.name,'parents':[folder['id']]},media_body=MediaFileUpload(str(file),mimetype=mime,resumable=True),fields='id,name').execute()
    print('Salvo:',saved['name'],saved['id'])
print('PASTA DE RESULTADOS: https://drive.google.com/drive/folders/'+folder['id'])

## Depois de executar
Diga no ChatGPT: **“EXP-06B executado; confira o Drive”**. A análise de traces é **diagnóstica**; nenhum resultado automático confirma mixer ou cria um novo holdout independente.